In [1]:
"""
Reddit Comment Labeling Pipeline (LM Studio, OpenAI-compatible API)
-------------------------------------------------------------------
- Input: dictionaries (Reddit-style with keys 'id' and 'body'), OR a dict mapping id->text,
         OR a pandas DataFrame with columns ['id','body'].
- Retries: Up to 3 attempts per (comment, task) with exponential backoff.
- Strict JSON validation against task schemas; otherwise retry.
- Output: NDJSON (one line per (comment_id, task)).
- Tasks implemented:
    - stance_intensity
    - epistemic_modality
    - justification_density
    - responsiveness
    - agreement
    - civility
    - sarcasm

Extensions:
    - For each comment we additionally run an argument extraction call:
        {
          "task": "argument_extraction",
          "arguments": [ "<string>", ... ],
          "confidence": <float in [0,1]>
        }
      The extracted arguments + confidence are added to each NDJSON line:
        "arguments": [...],
        "arguments_confidence": <float or null>,
        "arguments_error": <string or null>

    - Parent-ID of the comment is stored in the NDJSON output as "parent_id"
      (string or null). For DataFrame input this uses `parent_col` if available.
      For the generic run_pipeline (no DF), parent_id is always null.

Requirements:
    pip install requests pandas
    reddit data has to be stored locally in a folder called 'data' (optional demo below).

LM Studio:
    - Enable the local HTTP server in LM Studio (OpenAI-compatible API).
    - Default endpoint: http://localhost:1234/v1
    - Set MODEL_NAME to the exact local model identifier shown in LM Studio.
"""

import json
import time
import os
from typing import Dict, Any, List, Optional, Iterable, Tuple, Union, Set, Callable
from pathlib import Path

import pandas as pd
import requests



In [2]:
# ------------------------
# Configuration
# ------------------------

LMSTUDIO_BASE_URL = "http://127.0.0.1:1234"  # Change if LM Studio runs on a different host/port
MODEL_NAME = "ibm/granite-3.2-8b"               # e.g., "qwen2.5-7b-instruct"
TIMEOUT_SECONDS = 60                        # HTTP request timeout
MAX_RETRIES = 3                             # Max attempts per (comment, task)
RETRY_BACKOFF_SECONDS = 1.5                 # Exponential backoff base

# %%
# ------------------------
# Read reddit data (demo)
# ------------------------
p_sub = Path("data/submissions.ndjson")
p_com = Path("data/comments.ndjson")

print("Exists submissions:", p_sub.exists(), p_sub.resolve())
print("Exists comments   :", p_com.exists(), p_com.resolve())

if p_sub.exists():
    df_submissions = pd.read_json(p_sub, lines=True)
else:
    df_submissions = pd.DataFrame()

if p_com.exists():
    df_comments = pd.read_json(p_com, lines=True)
else:
    df_comments = pd.DataFrame()

# Minimal Demo-Subset: nur id und body
if not df_comments.empty:
    df_comments = (
        df_comments.loc[:, ["id", "body"]]
        .dropna(subset=["id", "body"])
        .astype({"id": "string", "body": "string"})
        .reset_index(drop=True)
    )
else:
    df_comments = pd.DataFrame(columns=["id", "body"])



Exists submissions: True C:\Users\rolfa\DataspellProjects\reddit\reddit-label\data\submissions.ndjson
Exists comments   : True C:\Users\rolfa\DataspellProjects\reddit\reddit-label\data\comments.ndjson


In [3]:
# ------------------------
# Task-Spezifikation
# ------------------------

TASK_SPECS: Dict[str, Dict[str, Any]] = {
    "stance_intensity": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "stance_intensity",
        "score_key": "score",
        "score_type": (int, float),   # numerisch, außer wenn "ABSTAIN"
        "confidence_range": (0.0, 1.0),
        "score_range": (1, 6),
        "allow_abstain": True,
        "label_key": None,
    },
    "epistemic_modality": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "epistemic_modality",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "justification_density": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "justification_density",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, float("inf")),  # nichtnegativ
        "allow_abstain": True,
        "label_key": None,
    },
    "responsiveness": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "responsiveness",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "agreement": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "agreement",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (-1.0, 1.0),
        "allow_abstain": True,
        "label_key": None,
    },
    "civility": {
        "schema_keys": {"task", "label", "confidence"},
        "task_value": "civility",
        "score_key": None,
        "label_key": "label",
        "label_type": (int,),         # int 1..6, außer wenn "ABSTAIN"
        "score_range": (1.0, 6.0),
        "confidence_range": (0.0, 1.0),
        "allow_abstain": True,
    },
    "sarcasm": {
        "schema_keys": {"task", "label", "confidence"},
        "task_value": "sarcasm",
        "score_key": None,
        "label_key": "label",
        "label_type": (int,),         # 0 oder 1, außer wenn "ABSTAIN"
        "score_range": (0.0, 1.0),    # 0 = kein Sarkasmus, 1 = Sarkasmus
        "confidence_range": (0.0, 1.0),
        "allow_abstain": True,
    },
}


In [4]:
# ------------------------
# System prompt (labeling tasks)
# ------------------------

SYSTEM_PROMPT = """You are a careful, literal discourse annotator. Use ONLY the provided TEXT,
PARENT_TEXT (if given), and TOPIC_DEF. Do not infer beyond the text or use any external knowledge.
Output must be strictly valid JSON matching the registered schema for the task. Include a "confidence" field in [0,1].

You MUST prefer returning "ABSTAIN" over guessing. If the evidence for a numeric/ordinal label is weak,
conflicting, or the text is very short, return "ABSTAIN" for that task.

English only. If TEXT has fewer than ~15 tokens, or evidence is weak/ambiguous, return "ABSTAIN"
instead of trying to infer a score/label. When you return "ABSTAIN", still provide a confidence score
in [0,1] reflecting your confidence that abstaining is appropriate (typically low, e.g., <= 0.4).

B.2 Instruction templates and schemas
All tasks share the same idea: return a single JSON object for the requested task.

IMPORTANT:
- For each task, the main numeric/ordinal field ("score" or "label") can either be:
  - a valid number in the specified range, OR
  - the string "ABSTAIN" (if you cannot reliably decide).
- Never output any other strings in place of numeric scores/labels.

--------------------------------------------------
Stance intensity (ordinal; [1,6])
--------------------------------------------------
TASK: Estimate the author’s stance toward TOPIC_DEF on a 1–6 scale
(1 = strongly against, 3–4 = neutral/unclear, 6 = strongly in favor).

Use "ABSTAIN" if:
- there is no clear evidence of stance toward TOPIC_DEF, or
- the text is very short or off-topic, or
- signals are too mixed/ambiguous to place on the scale.

JSON schema:
{
  "task": "stance_intensity",
  "score": <int in [1, 6] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Epistemic modality (continuous; [0,1])
--------------------------------------------------
TASK: Score epistemic modality: density of hedges, modal verbs, and explicit uncertainty,
normalized by length. Higher = more expressed humility/uncertainty.

If you cannot reliably judge (e.g., very short or unclear text), return "ABSTAIN".

JSON schema:
{
  "task": "epistemic_modality",
  "score": <float in [0.0,1.0] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Justification density (per 100 words)
--------------------------------------------------
TASK: Count distinct justification units (claim + warrant), normalized per 100 words.
If uncertain, or if there is not enough content to identify justification units, return "ABSTAIN".

JSON schema:
{
  "task": "justification_density",
  "score": <non-negative float OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Responsiveness (continuous; [0,1])
--------------------------------------------------
TASK: Rate how directly this reply addresses its {PARENT_TEXT} (semantic overlap/engagement).

If PARENT_TEXT is not provided, or if the relation between TEXT and PARENT_TEXT is unclear,
return "ABSTAIN".

JSON schema:
{
  "task": "responsiveness",
  "score": <float in [0.0,1.0] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Agreement (continuous; [-1,1])
--------------------------------------------------
TASK: Rate agreement with {PARENT_TEXT}.
-1 = contradicts, 0 = neutral/unrelated, 1 = fully agrees.

If PARENT_TEXT is not provided, or if agreement cannot be reliably determined (e.g., off-topic,
sarcastic or ambiguous content without clear polarity), return "ABSTAIN".

JSON schema:
{
  "task": "agreement",
  "score": <float in [-1.0,1.0] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Civility (ordinal 1–6)
--------------------------------------------------
TASK: Rate civility on 1 (highly uncivil/insulting) to 6 (highly civil).

If civility is hard to judge (e.g., context missing, mixed cues, or text too short),
return "ABSTAIN".

JSON schema:
{
  "task": "civility",
  "label": <integer 1..6 OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

--------------------------------------------------
Sarcasm (binary 0/1)
--------------------------------------------------
TASK: Detect whether the TEXT is sarcastic or clearly ironic regarding its main target or TOPIC_DEF.

- Use 1 if there is clear sarcastic or ironic intent (e.g., praise used to convey criticism,
  exaggerated contrast between words and obvious reality, well-known sarcastic formulae).
- Use 0 if the text is clearly non-sarcastic and literal.
- If signals are ambiguous, context is insufficient, or the text is too short to decide,
  return "ABSTAIN".

JSON schema:
{
  "task": "sarcasm",
  "label": <0 or 1 OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}
"""

# Template that tells the model which single task to execute now
TASK_INSTRUCTION_TEMPLATE = (
    "Now perform ONLY the task = {task_name}. "
    "Return strictly valid JSON for that task and nothing else (no Markdown). "
    "Ensure keys and value ranges match the schema exactly."
)

# ------------------------
# Argument-Extraktion Prompt
# ------------------------

ARG_SYSTEM_PROMPT = """You are an argument mining annotator.
Use ONLY the provided TEXT and PARENT_TEXT (if given). Do not infer beyond the text.

Your task: extract explicit argumentative units from TEXT.
An argument is a (claim + reason), possibly implicit, but grounded in the text.

Output strictly valid JSON:
{
  "task": "argument_extraction",
  "arguments": [ "<short description of argument 1>", "<short description of argument 2>", ... ],
  "confidence": <float in [0,1]>
}

Guidelines:
- "arguments" must be a list of short strings (summaries or spans).
- If TEXT contains no clear arguments, use an empty list [].
- Do NOT hallucinate content not justified by the text.
- Prefer under-detection (few arguments) over over-detection.
"""

ARG_USER_TEMPLATE = (
    "TEXT: {text}\n\n"
    "If available, PARENT_TEXT (context for replies): {parent_text}\n\n"
    "Now perform ONLY argument_extraction as described in the system prompt. "
    "Return strictly valid JSON and nothing else."
)


In [ ]:
# ------------------------
# HTTP call utilities
# ------------------------

def call_lmstudio_chat(messages: List[Dict[str, str]], temperature: float = 0.0) -> str:
    """
    Call LM Studio (OpenAI-compatible) Chat Completions API and return raw text.
    Raises requests.RequestException on network/HTTP errors.
    """
    url = f"{LMSTUDIO_BASE_URL}/v1/chat/completions"
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": temperature,
        "stream": False,
    }
    resp = requests.post(url, json=payload, timeout=TIMEOUT_SECONDS)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"].strip()


def _is_number(x: Any) -> bool:
    """Helper: check numeric types."""
    return isinstance(x, (int, float))


# ------------------------
# Response validation (label tasks)
# ------------------------

def validate_response(task: str, obj: Dict[str, Any]) -> Optional[str]:
    """
    Validate a single task response against TASK_SPECS[task].

    Supports:
    - "ABSTAIN" for score/label if allow_abstain=True
    """
    spec = TASK_SPECS[task]

    if not isinstance(obj, dict):
        return "Response is not a JSON object"

    missing = spec["schema_keys"] - set(obj.keys())
    if missing:
        return f"Missing required keys: {sorted(missing)}"

    if obj.get("task") != spec["task_value"]:
        return f'Field "task" must be "{spec["task_value"]}"'

    # confidence
    conf = obj.get("confidence")
    if not _is_number(conf):
        return '"confidence" must be a number'
    lo_c, hi_c = spec["confidence_range"]
    if not (lo_c <= conf <= hi_c):
        return f'"confidence" must be in [{lo_c}, {hi_c}]'

    label_key = spec.get("label_key")
    score_key = spec.get("score_key")

    # Label-based tasks (e.g. civility, sarcasm)
    if label_key is not None:
        val = obj.get(label_key)

        # ABSTAIN handling
        if isinstance(val, str):
            if spec.get("allow_abstain") and val == "ABSTAIN":
                return None
            return f'"{label_key}" must be an integer in range or "ABSTAIN"'

        label_type = spec.get("label_type", (int,))
        if not isinstance(val, label_type):
            return f'"{label_key}" has wrong type (expected {label_type})'

        lo, hi = spec["score_range"]
        if not (lo <= float(val) <= hi):
            return f'"{label_key}" out of range [{lo}, {hi}]'
        return None

    # Score-based tasks
    if score_key is not None:
        val = obj.get(score_key)

        if isinstance(val, str):
            if spec.get("allow_abstain") and val == "ABSTAIN":
                return None
            return f'"{score_key}" must be a number or "ABSTAIN"'

        if not _is_number(val):
            return f'"{score_key}" must be a number'
        lo, hi = spec["score_range"]
        if not (lo <= float(val) <= hi):
            return f'"{score_key}" out of range [{lo}, {hi}]'
        return None

    return None


# ------------------------
# Validation: Argument-Extraktion
# ------------------------

def validate_arguments(obj: Dict[str, Any]) -> Optional[str]:
    """
    Validate the argument_extraction response.
    Expected schema:
    {
      "task": "argument_extraction",
      "arguments": [ "<string>", ... ],
      "confidence": <float in [0,1]>
    }
    """
    if not isinstance(obj, dict):
        return "Response is not a JSON object"

    if obj.get("task") != "argument_extraction":
        return 'Field "task" must be "argument_extraction"'

    if "arguments" not in obj or "confidence" not in obj:
        return "Missing required keys: 'arguments' and/or 'confidence'"

    args = obj["arguments"]
    if not isinstance(args, list):
        return '"arguments" must be a list'
    for i, a in enumerate(args):
        if not isinstance(a, str):
            return f'"arguments[{i}]" must be a string'

    conf = obj["confidence"]
    if not _is_number(conf):
        return '"confidence" must be a number'
    if not (0.0 <= conf <= 1.0):
        return '"confidence" must be in [0,1]'

    return None


# ------------------------
# Prompt construction
# ------------------------

def build_user_prompt(task: str, text: str, parent_text: Optional[str] = None) -> str:
    """
    Compose the user message that includes TEXT, optional PARENT_TEXT, and the task directive.
    """
    parts = {"TEXT": text}
    if parent_text is not None and str(parent_text).strip():
        parts["PARENT_TEXT"] = str(parent_text)
    directive = TASK_INSTRUCTION_TEMPLATE.format(task_name=task)
    return json.dumps(parts, ensure_ascii=False) + "\n\n" + directive


def build_argument_prompt(text: str, parent_text: Optional[str] = None) -> str:
    """
    Compose the user message for argument_extraction.
    """
    pt = parent_text if (parent_text is not None and str(parent_text).strip()) else "N/A"
    return ARG_USER_TEMPLATE.format(text=text, parent_text=pt)


# ------------------------
# Single annotation with retries (label tasks)
# ------------------------

def annotate_one(task: str, text: str, parent_text: Optional[str] = None) -> Dict[str, Any]:
    """
    Run one (task, text[, parent_text]) through LM Studio with retries and strict JSON validation.
    Returns the valid task-JSON on success, or {"task": task, "error": "..."} after all retries fail.
    """
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(task, text, parent_text)},
            ]
            raw = call_lmstudio_chat(messages, temperature=0.0)

            obj = json.loads(raw)
            err = validate_response(task, obj)
            if err is None:
                return obj
            else:
                last_err = f"Schema validation failed (attempt {attempt}): {err}"

        except requests.RequestException as e:
            last_err = f"HTTP error (attempt {attempt}): {e}"

        except json.JSONDecodeError as e:
            last_err = f"JSON parse error (attempt {attempt}): {e}"

        time.sleep((RETRY_BACKOFF_SECONDS ** attempt))

    return {
        "task": task,
        "error": last_err or "Unknown error",
    }


# ------------------------
# Argument extraction with retries
# ------------------------

def extract_arguments(text: str, parent_text: Optional[str] = None) -> Dict[str, Any]:
    """
    Run argument_extraction for a given text (and optional parent_text) with retries.
    Returns either a valid argument_extraction JSON or an error object.

    On success:
    {
      "task": "argument_extraction",
      "arguments": [...],
      "confidence": float
    }

    On failure:
    {
      "task": "argument_extraction",
      "error": "..."
    }
    """
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            messages = [
                {"role": "system", "content": ARG_SYSTEM_PROMPT},
                {"role": "user", "content": build_argument_prompt(text, parent_text)},
            ]
            raw = call_lmstudio_chat(messages, temperature=0.0)

            obj = json.loads(raw)
            err = validate_arguments(obj)
            if err is None:
                return obj
            else:
                last_err = f"Argument schema validation failed (attempt {attempt}): {err}"

        except requests.RequestException as e:
            last_err = f"Argument HTTP error (attempt {attempt}): {e}"

        except json.JSONDecodeError as e:
            last_err = f"Argument JSON parse error (attempt {attempt}): {e}"

        time.sleep((RETRY_BACKOFF_SECONDS ** attempt))

    return {
        "task": "argument_extraction",
        "error": last_err or "Unknown error",
    }


# ------------------------
# NDJSON writer
# ------------------------

def write_ndjson_line(fp, obj: Dict[str, Any]) -> None:
    """Write a single JSON object as one NDJSON line."""
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")


# ------------------------
# Input normalization for generic pipeline
# ------------------------

NormalizedItem = Tuple[str, str]  # (comment_id, body_text)


def _normalize_input(
    comments: Union[
        Dict[str, Any],       # either a single reddit dict with keys 'id' & 'body' OR mapping id->text
        List[Dict[str, Any]], # list of reddit dicts
        pd.DataFrame,         # DataFrame with columns ['id','body']
    ]
) -> Iterable[NormalizedItem]:
    """
    Normalize different input shapes to an iterator of (comment_id, body_text).
    Accepted forms:
      - Single Reddit-style dict with 'id' and 'body'
      - List of Reddit-style dicts with 'id' and 'body'
      - Mapping {id: text}
      - pd.DataFrame with columns ['id','body']
    """
    if isinstance(comments, pd.DataFrame):
        if not {"id", "body"}.issubset(comments.columns):
            raise ValueError("DataFrame input must contain columns ['id','body'].")
        for _, row in comments.iterrows():
            cid = str(row["id"])
            text = str(row["body"])
            if text and cid:
                yield (cid, text)
        return

    if isinstance(comments, dict):
        if "id" in comments and "body" in comments:
            yield (str(comments["id"]), str(comments["body"]))
            return
        for k, v in comments.items():
            cid = str(k)
            text = str(v)
            if text and cid:
                yield (cid, text)
        return

    if isinstance(comments, list):
        for item in comments:
            if not isinstance(item, dict):
                raise ValueError("List input must contain dictionaries with keys ['id','body'].")
            if "id" not in item or "body" not in item:
                raise ValueError("Each dictionary in the list must have 'id' and 'body'.")
            yield (str(item["id"]), str(item["body"]))
        return

    raise TypeError(
        "Unsupported input type. Provide a DataFrame with ['id','body'], "
        "a dict mapping id->text, a single reddit dict with 'id'/'body', or a list of such dicts."
    )


# ------------------------
# Generic main pipeline (no parents, parent_id=None)
# ------------------------

def run_pipeline(
    comments: Union[Dict[str, Any], List[Dict[str, Any]], pd.DataFrame],
    tasks: Optional[List[str]] = None,
    ndjson_path: str = "labels.ndjson",
) -> None:
    """
    Run the labeling pipeline on generic input (no parent-text lookup).

    Output format (NDJSON):
        One line per (comment_id, task), with a JSON object:
        {
          "comment_id": "<id string>",
          "comment_index": <int>,
          "parent_id": null,
          "task": "<task_name>",
          "result": { ... },
          "arguments": [...],
          "arguments_confidence": <float or null>,
          "arguments_error": <string or null>
        }
    """
    if tasks is None:
        tasks = list(TASK_SPECS.keys())

    iterator = list(_normalize_input(comments))

    with open(ndjson_path, "w", encoding="utf-8") as f:
        for idx, (comment_id, text) in enumerate(iterator):
            # Argument-Extraktion einmal pro Kommentar
            arg_res = extract_arguments(text, parent_text=None)
            if "error" in arg_res:
                arguments = []
                arg_conf = None
                arg_err = arg_res["error"]
            else:
                arguments = arg_res.get("arguments", [])
                arg_conf = arg_res.get("confidence", None)
                arg_err = None

            for task in tasks:
                result = annotate_one(task, text, parent_text=None)
                out = {
                    "comment_id": comment_id,
                    "comment_index": idx,
                    "parent_id": None,
                    "task": task,
                    "result": result,
                    "arguments": arguments,
                    "arguments_confidence": arg_conf,
                    "arguments_error": arg_err,
                }
                write_ndjson_line(f, out)


# ------------------------
# Resume-Helper: bereits gelabelte (comment_id, task)
# ------------------------

def _load_done_pairs(ndjson_path: str) -> Set[Tuple[str, str]]:
    """
    Liest (comment_id, task)-Pairs aus einer bereits existierenden NDJSON,
    um doppelte Arbeit zu vermeiden (Resume-Funktionalität).
    """
    done: Set[Tuple[str, str]] = set()
    if not os.path.exists(ndjson_path):
        return done
    with open(ndjson_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                cid = str(obj.get("comment_id", ""))
                t = str(obj.get("task", ""))
                if cid and t:
                    done.add((cid, t))
            except Exception:
                continue
    return done


# ------------------------
# DataFrame-basierte Pipeline mit Parent-Text & Parent-ID
# ------------------------

def label_dataframe(
    df: pd.DataFrame,
    tasks: Optional[List[str]] = None,
    ndjson_path: str = "labels.ndjson",
    id_col: str = "id",
    text_col: str = "body",
    batch_size: int = 500,
    start_index: int = 0,
    end_index: Optional[int] = None,
    skip_existing: bool = True,
    progress: Optional[Callable[[int, int, Dict[str, Any]], None]] = None,
    parent_col: str = "parent_id",
) -> Dict[str, Any]:
    """
    Labelt Kommentare aus einem DataFrame in Batches und schreibt NDJSON Zeile-für-Zeile.

    Pro Kommentar wird zusätzlich vor den Label-Tasks eine Argument-Extraktion durchgeführt.
    Das Ergebnis dieser Argument-Extraktion wird in jeder Zeile mitgeführt.

    Rückgabe:
        dict mit Countern (processed_rows, written_records, skipped_records, errors).
    """
    assert id_col in df.columns, f"Spalte '{id_col}' fehlt im DataFrame."
    assert text_col in df.columns, f"Spalte '{text_col}' fehlt im DataFrame."

    if tasks is None:
        tasks = list(TASK_SPECS.keys())

    n_total = len(df)
    if end_index is None or end_index > n_total:
        end_index = n_total
    if start_index < 0:
        start_index = 0
    if start_index >= end_index:
        return {
            "processed_rows": 0,
            "written_records": 0,
            "skipped_records": 0,
            "errors": 0,
            "note": "Nichts zu verarbeiten (Start >= Ende).",
        }

    # Parent-Lookup (id -> text), falls parent_col existiert
    parent_lookup: Optional[Dict[str, str]] = None
    if parent_col in df.columns:
        id_series = df[id_col].astype(str)
        text_series = df[text_col].astype(str)
        parent_lookup = dict(zip(id_series, text_series))

    # Resume: bereits gelabelte (id, task)
    already_done: Set[Tuple[str, str]] = set()
    if skip_existing:
        already_done = _load_done_pairs(ndjson_path)

    written_records = 0
    skipped_records = 0
    error_count = 0
    processed_rows = 0

    with open(ndjson_path, "a", encoding="utf-8") as fp:
        for batch_start in range(start_index, end_index, batch_size):
            batch_end = min(batch_start + batch_size, end_index)
            batch = df.iloc[batch_start:batch_end]

            for local_idx, row in batch.iterrows():
                comment_id = str(row[id_col]) if pd.notna(row[id_col]) else ""
                text = str(row[text_col]) if pd.notna(row[text_col]) else ""

                # Parent-ID & -Text
                parent_id: Optional[str] = None
                parent_text: Optional[str] = None
                if parent_col in df.columns and pd.notna(row.get(parent_col, None)):
                    parent_id = str(row[parent_col])
                    if parent_lookup is not None and parent_id in parent_lookup:
                        parent_text = parent_lookup[parent_id]

                if not comment_id or not text or not text.strip():
                    skipped_records += len(tasks)
                    processed_rows += 1
                    if progress:
                        progress(processed_rows, end_index - start_index, {
                            "written_records": written_records,
                            "skipped_records": skipped_records,
                            "errors": error_count
                        })
                    continue

                # Argumente einmal pro Kommentar extrahieren
                arg_res = extract_arguments(text, parent_text=parent_text)
                if "error" in arg_res:
                    arguments = []
                    arg_conf = None
                    arg_err = arg_res["error"]
                else:
                    arguments = arg_res.get("arguments", [])
                    arg_conf = arg_res.get("confidence", None)
                    arg_err = None

                for task in tasks:
                    if skip_existing and (comment_id, task) in already_done:
                        skipped_records += 1
                        continue

                    try:
                        result = annotate_one(task, text, parent_text=parent_text)
                        out = {
                            "comment_id": comment_id,
                            "comment_index": int(local_idx),  # Original-Index im DF
                            "parent_id": parent_id,           # string oder None
                            "task": task,
                            "result": result,
                            "arguments": arguments,
                            "arguments_confidence": arg_conf,
                            "arguments_error": arg_err,
                        }
                        write_ndjson_line(fp, out)
                        written_records += 1
                        if written_records % 100 == 0:
                            fp.flush()
                    except Exception:
                        error_count += 1

                processed_rows += 1

                if progress:
                    progress(processed_rows, end_index - start_index, {
                        "written_records": written_records,
                        "skipped_records": skipped_records,
                        "errors": error_count
                    })

    return {
        "processed_rows": processed_rows,
        "written_records": written_records,
        "skipped_records": skipped_records,
        "errors": error_count,
    }


# %%

In [7]:
# Demo-Aufruf
if not df_comments.empty:
    n_entries = 200   # <== hier beliebig anpassen

    tasks = [
        "stance_intensity",
        "civility",
        "epistemic_modality",
        "justification_density",
        "responsiveness",
        "agreement",
        "sarcasm",
    ]

    df_subset = df_comments.iloc[:n_entries]

    def simple_progress(done, total, s):
        if done % 500 == 0 or done == total:
            print(f"[{done}/{total}] written={s['written_records']} skipped={s['skipped_records']} errors={s['errors']}")

    stats = label_dataframe(
        df=df_subset,
        tasks=tasks,
        ndjson_path=f"labels_{n_entries}.ndjson",
        batch_size=1000,
        skip_existing=True,
        progress=simple_progress,
        # parent_col="parent_id"  # ggf. anpassen, wenn du Parent-IDs mit drin hast
    )

    print(f"Labeling abgeschlossen ({n_entries} Einträge).")
    print(stats)
else:
    print("df_comments ist leer")


KeyboardInterrupt: 